# 005 — PSHA results: PGA / SA hazard curves and hazard surfaces

Post-processes the multi-period WP1 PSHA calculation (`SA_psha_eps4`) into hazard curves for
the 60 selected sites, and reshapes them into per-site **hazard surfaces**: one 2-D array per
statistic with **rows = IMLs** and **columns = periods**.

`004-psha_results.ipynb` does the same job for the AvgSA calculations, which carry a single
IMT. This calculation carries **20**: `PGA` plus `SA(0.05)`–`SA(1.20)`. Downstream work needs
the hazard at a structure's *exact* first-mode period, which is not one of the 20, so the
useful artifact is not 20 unrelated curves but an array that can be interpolated along its
period axis. This notebook produces those arrays and the `periods` vector that indexes them;
it deliberately does **not** interpolate — the choice of interpolation scheme belongs to the
consumer.

1. Resolves the OpenQuake `calc_id` of `SA_psha_eps4` from `wp1/psha_manifest.json` — no
   hardcoded integers.
2. Extracts the hazard curves from the datastore into the flat dictionary `004` uses
   (`curves[site_id][imt][stat] -> (n_iml, 2)`), in **mean annual frequency of exceedance
   (MAFE)**, or reloads them from the pickle where the datastore is not reachable (`SOURCE`).
3. Derives the `periods` vector from the IMT names and checks the shared IML grid.
4. Builds and saves the hazard surfaces.
5. Plots the hazard curves at all periods for one representative site per group.
6. Plots the 475- and 2475-year uniform hazard spectra for all 60 sites.

## The shared IML grid — an assumption, not a guarantee

The surfaces stack the MAFE columns of 20 separate hazard curves side by side against a
**single** `imls` vector. That is only meaningful because every IMT in this calculation was
given the same IML grid in the job file:

```
"PGA":      logscale(0.0005, 5.00, 25)
"SA(0.05)": logscale(0.0005, 5.00, 25)
...
"SA(1.20)": logscale(0.0005, 5.00, 25)
```

**OpenQuake does not require this.** `intensity_measure_types_and_levels` takes an independent
level list per IMT, and it is common practice to shift the grid with period (short periods
reach higher accelerations than long ones). A calculation built that way cannot be reshaped
this way without either carrying one IML vector per column — turning `imls` into its own
`(n_iml, n_period)` array — or resampling every curve onto a common grid first, which is an
interpolation and therefore a modelling choice.

Section 3 asserts the assumption against the datastore rather than trusting the job file. If
that assertion ever fires, the surfaces are the wrong data structure for that calculation and
the fix is one of the two options above, not a loosened tolerance.

## Prerequisites

The `SA_psha_eps4` entry must exist in `hazard_models/eshm20/wp1/psha_manifest.json`, and the
corresponding OpenQuake datastore (`calc_<id>.hdf5`) must be present in the local `oqdata`
directory.

The datastores are **machine-local and not tracked** (they are large and rebuildable). The
manifest is the git-tracked pointer that makes them reproducible — if the datastore is
missing, re-run the calculation rather than editing the calc id here.

To re-run only the reshaping and the plots without the datastore, `dvc pull` the curves pickle
and leave `SOURCE = "auto"` (or force `SOURCE = "pickle"`). Sections 3–6 need nothing else:
everything downstream reads `curves`, not the datastore.

## Dependencies

**Upstream:** `001-site_selection.ipynb` → `results/01_site_selection/sites.csv` (the site
register, and the source of the `seismicity` / `region` grouping used in the plots);
`hazard_models/eshm20/wp1/config_SA_psha_eps4.ini` and its manifest entry.

**Downstream:** anything needing spectral hazard at an arbitrary period — the per-site hazard
surfaces are the input to that interpolation.

## Outputs

Two pickles in `data_processed/03_site_hazard` (**DVC-tracked**, like the rest of that folder):

```
SA_hazard_curves_60sites_4sig.pickle     curves[site_id][imt][stat] -> (n_iml, 2)
                                         col 0 = IML [g], col 1 = MAFE [1/yr]

SA_hazard_surfaces_60sites_4sig.pickle   {"imls":    (n_iml,)     IML grid [g], the row index
                                          "periods": (n_period,)  period [s], the column index
                                          "imts":    [str]        OQ IMT names, aligned with periods
                                          "sites":   {site_id: {stat: (n_iml, n_period) MAFE}}}
```

`PGA` is carried as period `0.0` and sorts first.

In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [ ]:
import string
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from matplotlib import colormaps
from matplotlib.colors import Normalize

from openquake.commonlib.datastore import read
from openquake.hazardlib.imt import from_string

from phd_project.config import config
from phd_project.plotting.plotting import custom_log_formatter
from phd_project.scripts.oqhelpers import get_hcurves_from_dstore
from phd_project.scripts import oq_runner
from phd_project.scripts.hazard import get_mask

cfg = config.load_config()

In [ ]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
IM = "SA"                       # IM family; also the prefix of the output pickle names
EPS, SIG = "eps4", "4sig"       # GMM truncation: manifest suffix -> filename suffix
ANALYSIS = f"{IM}_psha_{EPS}"   # -> "SA_psha_eps4", the manifest key

STAT = "mean"   # the statistic plotted (the pickles keep all of them)
STATS = ("mean", "quantile-0.16", "quantile-0.5", "quantile-0.84")

# The IML grid every IMT in this calculation shares, from config_SA_psha_eps4.ini:
#   intensity_measure_types_and_levels = {..., "SA(x)": logscale(0.0005, 5.00, 25), ...}
# OpenQuake's logscale(a, b, n) is np.logspace(log10(a), log10(b), n). Section 3 checks
# this against the datastore rather than taking the job file's word for it.
IMLS = np.logspace(np.log10(0.0005), np.log10(5.00), 25)

N_SITES = 60
REGIONS = list(range(6))
SEISMICITIES = ("high", "lowmod")

# Return periods for the uniform hazard spectra.
UHS_RTPS = (475, 2475)

HAZ_DIR = cfg["proc_data"]["site_hazard"]
MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_psha_manifest"]
SITES_FP = cfg["results"]["selected_sites_csv"]
SITE_MODEL_FP = cfg["hazard_models"]["eshm20_wp1_site_model"]
CURVES_FP = cfg["proc_data"]["SA_hazard_curves_4sig"]
SURFACES_FP = cfg["proc_data"]["SA_hazard_surfaces_4sig"]

# SOURCE: where the hazard curves come from.
#   "datastore" - extract from the OpenQuake datastore; fail if it is missing.
#   "pickle"    - reload the pickle a previous run wrote, ignoring the datastore.
#   "auto"      - datastore if it can be read, pickle if it cannot.
# The datastore is machine-local, so "auto" is what makes the rest of the notebook
# runnable anywhere the DVC-tracked pickle has been pulled.
SOURCE = "auto"

# SAVE: write the pickles. False = extract and plot only, touching nothing on disk.
# Only curves read from the datastore are written back; a pickle is never re-pickled.
SAVE = True
# -----------------------------------------------------------------------------

assert SOURCE in ("datastore", "pickle", "auto"), f"unknown SOURCE {SOURCE!r}"

print(f"hazard dir: {HAZ_DIR}")
print(f"manifest:   {MANIFEST_FP}")
print(f"analysis:   {ANALYSIS}")
print(f"IML grid:   {len(IMLS)} levels, {IMLS[0]:.4g} to {IMLS[-1]:.4g} g")
print(f"SOURCE={SOURCE}, SAVE={SAVE}")

## 1. Sites & calculation id

`sites.csv` and `wp1/site_model_casestudy_sites.csv` are row-aligned by construction (`003`
writes the latter from the former), so they can be joined positionally. The resulting index
**is** the site id used everywhere in this project.

The two grouping keys come from `sites.csv`: `seismicity` (`"high"` / `"lowmod"`) and `region`
(`0`–`5`) — 5 sites in each of the 12 combinations.

In [ ]:
sel_sites = pd.read_csv(SITES_FP)
site_model = pd.read_csv(SITE_MODEL_FP)
site_metadata = pd.concat([sel_sites, site_model], axis=1).T.drop_duplicates().T

assert len(site_metadata) == N_SITES, (
    f"expected {N_SITES} sites, got {len(site_metadata)} — "
    f"check {SITES_FP} and {SITE_MODEL_FP} are the same 60 rows in the same order")

print(site_metadata.groupby(["seismicity", "region"]).size().to_string())

In [ ]:
calc_ids = oq_runner.load_calc_ids(MANIFEST_FP) if MANIFEST_FP.exists() else {}

# The manifest is only needed to reach the datastore. Running from the pickle does
# not need it, so a missing entry is only fatal when SOURCE demands one.
assert not (ANALYSIS not in calc_ids and SOURCE == "datastore"), (
    f"{ANALYSIS} is not in {MANIFEST_FP} — run the calculation first, "
    f"or set SOURCE = 'pickle'")

if ANALYSIS in calc_ids:
    print(f"{ANALYSIS:16s} calc_id = {calc_ids[ANALYSIS]}")
else:
    print(f"no calc id for {ANALYSIS} — the curves will be read from the pickle")

## 2. Extract & save hazard curves

`get_hcurves_from_dstore` converts the OpenQuake probability of exceedance to MAFE
(`-ln(1 - poe) / investigation_time`), so the curves are in MAFE as stored. It already loops
over every IMT in the datastore, so it needs no change for a 20-IMT calculation — the result
is the same `curves[site_id][imt][stat]` shape `004` produces, with 20 IMT keys instead of 1.

The datastore is machine-local, so there are two possible sources: the OpenQuake datastore, or
the pickle a previous run wrote to `data_processed/03_site_hazard`. `SOURCE` picks between
them — `"auto"` (the default) tries the datastore and falls back to the pickle, so the
notebook runs anywhere the DVC-tracked pickle has been pulled. Only curves that came from a
datastore are written back; re-pickling a pickle would rewrite the file (and dirty DVC) for no
gain.

In [ ]:
def curves_from_datastore():
    """Extract the hazard curves from the OpenQuake datastore."""
    dstore = read(calc_ids[ANALYSIS])
    return get_hcurves_from_dstore(dstore, mafe=True)


def curves_from_pickle():
    """Reload the hazard curves from the pickle a previous run wrote."""
    with open(CURVES_FP, "rb") as f:
        return pickle.load(f)


source = "pickle"

if SOURCE in ("datastore", "auto"):
    try:
        curves = curves_from_datastore()
        source = "datastore"
    # openquake signals a missing calculation with several unrelated types
    # (dbapi.NotFound, KeyError, FileNotFoundError, h5py OSError), none of which
    # share a base class, so the fallback catches broadly.
    except Exception as exc:
        if SOURCE == "datastore":
            raise
        print(f"{ANALYSIS}: datastore unavailable "
              f"({type(exc).__name__}: {exc}) — reading the pickle instead")

if source == "pickle":
    curves = curves_from_pickle()

assert len(curves) == N_SITES, f"{source} has {len(curves)} sites, expected {N_SITES}"

if SAVE and source == "datastore":
    with open(CURVES_FP, "wb") as f:
        pickle.dump(curves, f)
    print(f"wrote {CURVES_FP.name}")

n_imts = len(curves[0])
n_stats = len(curves[0][next(iter(curves[0]))])
print(f"\nhazard curves loaded from the {source}: "
      f"{len(curves)} sites x {n_imts} IMTs x {n_stats} stats")

In [ ]:
# Structure check: the flat, MAFE shape 004 established, now with many IMTs.
assert sorted(curves) == list(range(N_SITES)), f"site keys are not 0..{N_SITES - 1}"

imts_present = list(curves[0])
for site_id, imt_curves in curves.items():
    assert list(imt_curves) == imts_present, f"site {site_id}: IMT keys differ from site 0"
    for imt, stat_curves in imt_curves.items():
        for stat in STATS:
            assert stat in stat_curves, f"site {site_id}/{imt}: stat {stat!r} missing"
            hc = stat_curves[stat]
            assert hc.ndim == 2 and hc.shape[1] == 2, (
                f"site {site_id}/{imt}/{stat}: curve shape {hc.shape}, expected (n, 2)")
            # Non-strict: the MAFE column flattens once it underflows to 0 at the
            # truncation ceiling.
            assert (np.diff(hc[:, 1]) <= 0).all(), (
                f"site {site_id}/{imt}/{stat}: MAFE column is not decreasing")

print("OK — curves[site_id][imt][stat] -> (n_iml, 2), col 0 = IML [g], col 1 = MAFE [1/yr]")
print(f"IMTs:  {imts_present}")
print(f"stats: {list(curves[0][imts_present[0]])}")

## 3. Periods & the shared IML grid

The period of each column comes from the IMT name itself, via OpenQuake's own parser
(`from_string("SA(0.35)").period -> 0.35`) rather than a regex — `PGA` resolves to `0.0` and
therefore sorts first. The datastore's IMT ordering is not guaranteed to match the job file's,
so the columns are sorted by period explicitly here, and that order is what `periods` and
`imts` record.

This is also where the shared-IML-grid assumption described at the top of the notebook is
checked: every one of the 20 × 4 (IMT, stat) curves at every site must carry the same IML
column, and that column must be the `IMLS` grid declared in the parameters. A failure here
means the surfaces are the wrong structure for this calculation — see the header for what to
do instead.

In [ ]:
imts = sorted(imts_present, key=lambda s: from_string(s).period)
periods = np.array([from_string(s).period for s in imts])

assert (np.diff(periods) > 0).all(), f"periods are not strictly increasing: {periods}"

# The assumption the surfaces rest on: one IML grid, shared by every IMT and stat.
for site_id in curves:
    for imt in imts:
        for stat in STATS:
            assert np.allclose(curves[site_id][imt][stat][:, 0], IMLS), (
                f"site {site_id}/{imt}/{stat}: IML column differs from the shared grid "
                f"IMLS — this calculation cannot be reshaped into a hazard surface "
                f"against a single IML vector (see the notebook header)")

print(f"{len(imts)} IMTs, periods [s]:")
print("  " + ", ".join(f"{imt}={T:g}" for imt, T in zip(imts, periods)))
print("\nPGA is carried as period 0.0 and sorts first.")
print(f"shared IML grid confirmed for all "
      f"{N_SITES} x {len(imts)} x {len(STATS)} curves.")

## 4. Build the hazard surfaces

One array per (site, stat): the MAFE columns of the 20 hazard curves stacked side by side, in
ascending period order.

```
surfaces["sites"][site_id][stat][i, j] = MAFE of exceeding IML surfaces["imls"][i]
                                         at period    surfaces["periods"][j]
```

`imls` and `periods` are the row and column indices; `imts` keeps the original OpenQuake IMT
names alongside `periods` so a column can be traced back to its source curve. Interpolating
along the period axis is left to the consumer — the scheme (linear vs log in period, how PGA
at `T = 0` is treated) is a modelling choice, not a property of these data.

In [ ]:
surfaces = {
    "imls": IMLS,        # (n_iml,)    row index, IML [g]
    "periods": periods,  # (n_period,) column index, period [s], ascending, PGA = 0.0
    "imts": imts,        # OQ IMT names, aligned with `periods`
    "sites": {
        site_id: {
            stat: np.column_stack([curves[site_id][imt][stat][:, 1] for imt in imts])
            for stat in STATS
        }
        for site_id in sorted(curves)
    },
}

expected_shape = (len(IMLS), len(periods))
for site_id, stat_surfaces in surfaces["sites"].items():
    for stat, surface in stat_surfaces.items():
        assert surface.shape == expected_shape, (
            f"site {site_id}/{stat}: surface {surface.shape}, expected {expected_shape}")
        assert (np.diff(surface, axis=0) <= 0).all(), (
            f"site {site_id}/{stat}: MAFE does not decrease with IML in every column")

# The columns must be the curves they came from, not a transposed or misaligned view.
j = imts.index("SA(1.00)") if "SA(1.00)" in imts else len(imts) - 1
assert np.array_equal(surfaces["sites"][0][STAT][:, j],
                      curves[0][imts[j]][STAT][:, 1]), "column/curve misalignment"

if SAVE and source == "datastore":
    with open(SURFACES_FP, "wb") as f:
        pickle.dump(surfaces, f)
    print(f"wrote {SURFACES_FP.name}")

print(f"{len(surfaces['sites'])} sites x {len(STATS)} stats, "
      f"each {expected_shape[0]} IMLs x {expected_shape[1]} periods")

## 5. Plotting constants & helpers

`uhs` is the only new piece: a uniform hazard spectrum is the IML at a fixed MAFE, read off
each period column of a surface. That is an inverse interpolation down the column, done in
log-log space to match how the curves are plotted.

Columns whose MAFE range does not bracket the target return `nan` rather than an extrapolated
value. This is expected at long periods and long return periods — the 4σ GMM truncation puts a
ceiling on the ground motion the model will produce, so the curve simply stops before the
target MAFE is reached. A `nan` at 475 years, on the other hand, would mean something is wrong
with the calculation.

In [ ]:
legend_font_params = {'size': 9}
legend_title_font_params = {'weight': 'bold', 'size': 9}

# high vs low/moderate seismicity, as in 004
seismicity_colours = {"high": "r", "lowmod": "g"}
seismicity_labels = {"high": "High seismicity", "lowmod": "Low/moderate seismicity"}

# period ramp for the multi-period hazard curve panels
period_cmap = colormaps["viridis"]

In [ ]:
def group_site_ids(metadata, seismicity, region):
    """Site ids belonging to one (seismicity, region) group, in site-id order."""
    mask = get_mask(["seismicity", "region"], [seismicity, region], metadata)
    return sorted(metadata[mask].index.tolist())


def uhs(surface, mafe):
    """Uniform hazard spectrum of one surface at `mafe` -> ndarray (n_period,).

    Inverse-interpolates each period column in log-log space: the IML at the
    requested mean annual frequency of exceedance. Columns whose usable MAFE range
    does not bracket `mafe` give nan rather than an extrapolated value — at the top
    end that is the GMM truncation ceiling, where the curve stops.
    """
    imls = np.full(surface.shape[1], np.nan)

    for j, column in enumerate(surface.T):
        usable = column > 0
        if usable.sum() < 2:
            continue

        m, x = column[usable], IMLS[usable]
        if not (m.min() <= mafe <= m.max()):
            continue

        # np.interp needs an ascending x, and MAFE descends with IML.
        imls[j] = np.exp(np.interp(np.log(mafe), np.log(m[::-1]), np.log(x[::-1])))

    return imls


def style_hc_axis(ax, panel_idx, region):
    """The shared hazard-curve axis furniture (as 004)."""
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(custom_log_formatter))
    ax.set_xlim(0.001, 5)
    ax.set_ylim(1e-6, 1)

    ax.grid(True, which="both", ls="-.", color="0.8")
    ax.minorticks_on()
    ax.tick_params(axis='y', which='minor', left=False)

    ax.text(0.0011, 1.5e-6, f"({string.ascii_uppercase[panel_idx]})")
    ax.text(0.5, 2e-1, f"Region {region}")


def style_uhs_axis(ax, panel_idx, region):
    """Uniform hazard spectrum axis furniture: log Sa against linear period."""
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_log_formatter))
    ax.set_xlim(-0.05, periods.max() + 0.05)
    # 5 g is the top of the IML grid, so no spectrum can run off the top. The floor
    # is low enough to keep the long-period end of the low-seismicity sites on the
    # axis rather than silently clipping it.
    ax.set_ylim(1e-3, 5)

    ax.grid(True, which="both", ls="-.", color="0.8")
    ax.minorticks_on()
    ax.tick_params(axis='x', which='minor', bottom=False)

    ax.text(0.0, 1.3e-3, f"({string.ascii_uppercase[panel_idx]})")
    ax.text(0.0, 3.0, f"Region {region}")

## 6. Hazard curves at all periods — one site per group

One figure per seismicity class, 2×3 region panels. Each panel shows **a single site** — the
first site id of that (seismicity, region) group — with all 20 period curves drawn on it,
coloured along a `viridis` ramp from PGA to `SA(1.20)`. Plotting one site per panel is what
keeps 20 curves legible; the spread across the 5 sites in a group is what section 7 shows.

The expected behaviour: the curves fan out to the right up to roughly the plateau of the
spectrum, then march back to the left as the period grows and the spectral acceleration falls
away. The sharp drop at the right-hand end of each curve is the 4σ GMM truncation ceiling.

In [ ]:
norm = Normalize(vmin=periods.min(), vmax=periods.max())

for s in SEISMICITIES:
    fig, axs = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(9, 6))

    for ii, (r, ax) in enumerate(zip(REGIONS, axs.flatten())):
        site_id = group_site_ids(site_metadata, s, r)[0]

        for imt, T in zip(imts, periods):
            hc = curves[site_id][imt][STAT]
            pos = hc[:, 1] > 0      # log axes cannot show MAFE that underflowed to 0
            ax.loglog(hc[pos, 0], hc[pos, 1], color=period_cmap(norm(T)), lw=1.2)

        style_hc_axis(ax, ii, r)
        ax.text(0.0011, 4e-6, f"Site {site_id}", fontsize=8)

    axs[0, 0].set_ylabel("MAFE [1/yr]")
    axs[1, 0].set_ylabel("MAFE [1/yr]")
    for ax in axs[1, :]:
        ax.set_xlabel("Sa [g]")

    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=period_cmap),
                        ax=axs, fraction=0.03, pad=0.02)
    cbar.set_label("Period [s]  (0 = PGA)")

    fig.suptitle(f"SA hazard curves at all {len(imts)} periods - "
                 f"{seismicity_labels[s].lower()} sites "
                 f"({SIG[0]}$\\sigma$ truncation)\n"
                 f"one representative site per region")

## 7. Uniform hazard spectra

One figure per return period, 2×3 region panels, **all 60 sites**: the 5 high-seismicity sites
of each region in red and the 5 low/moderate sites in green. Each line is one site's IML read
off its hazard surface at MAFE = 1/RTP, at all 20 periods.

`PGA` is drawn at period 0, so the first segment of each line is the PGA-to-`SA(0.05)` step and
should not be read as a spectral shape. Gaps in a line are periods where that site's curve
never reaches the target MAFE (see `uhs`); at 2475 years some columns may drop out for that
reason. The final cell counts them.

In [ ]:
for rtp in UHS_RTPS:
    target_mafe = 1 / rtp

    fig, axs = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(9, 6))

    for ii, (r, ax) in enumerate(zip(REGIONS, axs.flatten())):
        for s in SEISMICITIES:
            for site_id in group_site_ids(site_metadata, s, r):
                spectrum = uhs(surfaces["sites"][site_id][STAT], target_mafe)
                ax.plot(periods, spectrum,
                        color=seismicity_colours[s], lw=1.2, alpha=0.8,
                        marker="o", ms=2.5)

        style_uhs_axis(ax, ii, r)

    axs[0, 0].set_ylabel("Sa [g]")
    axs[1, 0].set_ylabel("Sa [g]")
    for ax in axs[1, :]:
        ax.set_xlabel("Period [s]  (0 = PGA)")

    handles = [
        mlines.Line2D([], [], color=seismicity_colours[s], label=seismicity_labels[s])
        for s in SEISMICITIES
    ]
    leg = fig.legend(handles=handles, ncols=2,
                     loc="upper left", bbox_to_anchor=(0.075, 0.945),
                     prop=legend_font_params, frameon=False)
    leg._legend_box.align = "left"

    fig.suptitle(f"Uniform hazard spectra - {rtp} year return period, all 60 sites "
                 f"({SIG[0]}$\\sigma$ truncation)")
    fig.tight_layout(rect=(0, 0, 1, 0.92))

In [ ]:
# How much of each spectrum the truncation ceiling removes, per return period.
for rtp in UHS_RTPS:
    spectra = np.vstack([uhs(surfaces["sites"][site_id][STAT], 1 / rtp)
                         for site_id in sorted(surfaces["sites"])])
    missing = np.isnan(spectra)
    print(f"{rtp:5d} yr: {missing.sum():4d} / {spectra.size} (site, period) points beyond "
          f"the curve's range, affecting {missing.any(axis=1).sum()} of {N_SITES} sites")